# Silver — Budgets

Lê Delta Bronze e aplica qualidade de dados.

**Transformações:**
- Filtra `amount_usd > 0`, `month BETWEEN 1-12`, `provider` válido
- Deduplica por `(team_id, year, month, provider)`

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
spark = create_spark_session("silver_budgets")

# Processa TODOS os meses presentes no bronze (full reprocess)
df = spark.read.format("delta").load("s3a://datalake/bronze/budgets/")

df_silver = (
    df
    .filter(F.col("team_id").isNotNull())
    .filter(F.col("amount_usd") > 0)
    .filter(F.col("month").between(1, 12))
    .filter(F.col("provider").isin("AWS", "GCP", "Azure"))
    .dropDuplicates(["team_id", "year", "month", "provider"])
    .drop("_ingested_at", "_source_layer")
    .withColumn("_processed_at", F.current_timestamp())
)

output_path = "s3a://datalake/silver/budgets/"
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(output_path)
)

print(f"[silver_budgets] {df_silver.count()} budgets → {output_path}")
df_silver.groupBy("provider").agg(
    F.count("*").alias("n_budgets"),
    F.round(F.sum("amount_usd"), 2).alias("total_budget_usd")
).show()
spark.stop()
